In [1]:
import numpy as np
import os
from scipy.io import savemat, loadmat
from scipy.stats import qmc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
#from ES_MDA import ES_MDA
import pandas as pd
#import utility
#import forward_model
import sobol_seq
import copy
from write_script import write_script_site1 as write_script
from write_script import write_script_site2 as write_script2

### Site 1

In [2]:
def forward1(Num_ens):
    data=[]
    for i in range(Num_ens):
        res=pd.read_csv('Site1_param'+str(i+1)+'/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:233]
        A=np.hstack(((res.iloc[988,2:]-res.iloc[7714,2:]).values*1000,(res.iloc[2732,2:]-res.iloc[7714,2:]).values*1000,
                    (res.iloc[6917,2:]-res.iloc[7714,2:]).values*1000))
        data.append(A)
    return np.vstack(data)

In [3]:
#params= [bedrock permeability, top soil permebility, qv_sat, resistivity of top soil]

In [4]:
tol = 1e-12;
lambd = 1e-3;
V_obs=np.loadtxt('data/Site1_obs.txt').T
init=np.log10(np.array([1.58e-12,6.734728e-14,40,800]))
# set upper bound for all parameters
para_u = np.array([-10, -12, 4,3.5])
# set lower bound for all parameters
para_l = np.array([-16, -16, -1,1])
B=np.log((init - para_l) / (para_u - init))
#recover init: para_l + (para_u-para_l) * (np.exp(kk) / (1 + np.exp(kk)))

In [5]:
write=10**(para_l + (para_u-para_l) * (np.exp(B) / (1 + np.exp(B))))
write_script(write[0],write[1],101)
np.savetxt('Site1_param101.txt',write)

In [ ]:
res=pd.read_csv('Site1_param101/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:233]
a=(res.iloc[988,2:]-res.iloc[7714,2:]).values*1000
b=(res.iloc[2732,2:]-res.iloc[7714,2:]).values*1000
c=(res.iloc[6917,2:]-res.iloc[7714,2:]).values*1000
V_model=np.hstack((a,b,c)).reshape(-1,1)

In [ ]:
r = V_model - V_obs.reshape(-1,1);
cost = np.linalg.norm(r)**2
print(f"Initial Cost = {cost:e}")

In [6]:
# Compute the numerical Jacobian J (size: len(V_obs) x len(B))
nB = len(B)
nV = len(V_obs)
J = np.zeros((nV, nB))
delta = B * 0.2 

In [7]:
for i in range (nB):
    B_pertubed=B.copy()
    B_pertubed[i]=B[i]+delta[i]
    write=10**(para_l + (para_u-para_l) * (np.exp(B_pertubed) / (1 + np.exp(B_pertubed))))
    write_script(write[0],write[1],i+1)
    np.savetxt('Site1_param'+str(i+1)+'.txt',write)

In [ ]:
perturbed=forward1(nB).T
J=(perturbed-V_model) / delta

In [ ]:
dB = -np.linalg.solve(J.T @ J + lambd * np.eye(nB), J.T @ r)

In [ ]:
B = B + dB.reshape(-1,);
write=10**(para_l + (para_u-para_l) * (np.exp(B) / (1 + np.exp(B))))
write_script(write[0],write[1],101)
np.savetxt('Site1_param101.txt',write)

In [ ]:
res=pd.read_csv('Site1_param101/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:233]
a=(res.iloc[988,2:]-res.iloc[7714,2:]).values*1000
b=(res.iloc[2732,2:]-res.iloc[7714,2:]).values*1000
c=(res.iloc[6917,2:]-res.iloc[7714,2:]).values*1000
V_model=np.hstack((a,b,c)).reshape(-1,1)

In [ ]:
plt.plot(V_model)
plt.plot(V_obs)

In [ ]:
r = V_model - V_obs.reshape(-1,1);
cost = np.linalg.norm(r)**2
print(f"Cost iter 2= {cost:e}")

In [ ]:
# Compute the numerical Jacobian J (size: len(V_obs) x len(B))
nB = len(B)
nV = len(V_obs)
J = np.zeros((nV, nB))
delta = B * 0.1 

In [ ]:
for i in range (nB):
    B_pertubed=B.copy()
    B_pertubed[i]=B[i]+delta[i]
    write=10**(para_l + (para_u-para_l) * (np.exp(B_pertubed) / (1 + np.exp(B_pertubed))))
    write_script_site1(write[0],write[1],i+1)
    np.savetxt('Site1_param'+str(i+1)+'.txt',write)

In [ ]:
perturbed=forward1(nB).T
J=(perturbed-V_model) / delta

In [ ]:
dB = -np.linalg.solve(J.T @ J + lambd * np.eye(nB), J.T @ r)

In [ ]:
B = B + dB.reshape(-1,);
write=10**(para_l + (para_u-para_l) * (np.exp(B) / (1 + np.exp(B))))
write_script_site1(write[0],write[1],101)
np.savetxt('Site1_param101.txt',write)

In [ ]:
res=pd.read_csv('Site1_param101/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:233]
a=(res.iloc[988,2:]-res.iloc[7714,2:]).values*1000
b=(res.iloc[2732,2:]-res.iloc[7714,2:]).values*1000
c=(res.iloc[6917,2:]-res.iloc[7714,2:]).values*1000
V_model=np.hstack((a,b,c)).reshape(-1,1)

In [ ]:
plt.plot(V_model)
plt.plot(V_obs)

In [ ]:
r = V_model - V_obs.reshape(-1,1);
cost = np.linalg.norm(r)**2
print(f"Cost iter 2= {cost:e}")

In [ ]:
# Compute the numerical Jacobian J (size: len(V_obs) x len(B))
nB = len(B)
nV = len(V_obs)
J = np.zeros((nV, nB))
delta = B * 0.1 

In [ ]:
for i in range (nB):
    B_pertubed=B.copy()
    B_pertubed[i]=B[i]+delta[i]
    write=10**(para_l + (para_u-para_l) * (np.exp(B_pertubed) / (1 + np.exp(B_pertubed))))
    write_script_site1(write[0],write[1],i+1)
    np.savetxt('Site1_param'+str(i+1)+'.txt',write)

In [ ]:
perturbed=forward1(nB).T
J=(perturbed-V_model) / delta

In [ ]:
dB = -np.linalg.solve(J.T @ J + lambd * np.eye(nB), J.T @ r)

In [ ]:
B = B + dB.reshape(-1,);
write=10**(para_l + (para_u-para_l) * (np.exp(B) / (1 + np.exp(B))))
write_script_site1(write[0],write[1],101)
np.savetxt('Site1_param101.txt',write)

In [ ]:
res=pd.read_csv('Site1_param101/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:233]
a=(res.iloc[988,2:]-res.iloc[7714,2:]).values*1000
b=(res.iloc[2732,2:]-res.iloc[7714,2:]).values*1000
c=(res.iloc[6917,2:]-res.iloc[7714,2:]).values*1000
V_model=np.hstack((a,b,c)).reshape(-1,1)

In [ ]:
plt.plot(V_model)
plt.plot(V_obs)

In [ ]:
r = V_model - V_obs.reshape(-1,1);
cost = np.linalg.norm(r)**2
print(f"Cost iter 2= {cost:e}")

In [ ]:
# Compute the numerical Jacobian J (size: len(V_obs) x len(B))
nB = len(B)
nV = len(V_obs)
J = np.zeros((nV, nB))
delta = B * 0.1 

In [ ]:
for i in range (nB):
    B_pertubed=B.copy()
    B_pertubed[i]=B[i]+delta[i]
    write=10**(para_l + (para_u-para_l) * (np.exp(B_pertubed) / (1 + np.exp(B_pertubed))))
    write_script_site1(write[0],write[1],i+1)
    np.savetxt('Site1_param'+str(i+1)+'.txt',write)

In [ ]:
perturbed=forward1(nB).T
J=(perturbed-V_model) / delta

In [ ]:
dB = -np.linalg.solve(J.T @ J + lambd * np.eye(nB), J.T @ r)

In [ ]:
B = B + dB.reshape(-1,);
write=10**(para_l + (para_u-para_l) * (np.exp(B) / (1 + np.exp(B))))
write_script_site1(write[0],write[1],101)
np.savetxt('Site1_param101.txt',write)

In [ ]:
res=pd.read_csv('Site1_param101/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:233]
a=(res.iloc[988,2:]-res.iloc[7714,2:]).values*1000
b=(res.iloc[2732,2:]-res.iloc[7714,2:]).values*1000
c=(res.iloc[6917,2:]-res.iloc[7714,2:]).values*1000
V_model=np.hstack((a,b,c)).reshape(-1,1)

In [ ]:
plt.plot(V_model)
plt.plot(V_obs)

In [ ]:
r = V_model - V_obs.reshape(-1,1);
cost = np.linalg.norm(r)**2
print(f"Cost iter 2= {cost:e}")

In [ ]:
# Compute the numerical Jacobian J (size: len(V_obs) x len(B))
nB = len(B)
nV = len(V_obs)
J = np.zeros((nV, nB))
delta = B * 0.1 

In [ ]:
for i in range (nB):
    B_pertubed=B.copy()
    B_pertubed[i]=B[i]+delta[i]
    write=10**(para_l + (para_u-para_l) * (np.exp(B_pertubed) / (1 + np.exp(B_pertubed))))
    write_script_site1(write[0],write[1],i+1)
    np.savetxt('Site1_param'+str(i+1)+'.txt',write)

In [ ]:
perturbed=forward1(nB).T
J=(perturbed-V_model) / delta

In [ ]:
dB = -np.linalg.solve(J.T @ J + lambd * np.eye(nB), J.T @ r)

In [ ]:
B = B + dB.reshape(-1,);
write=10**(para_l + (para_u-para_l) * (np.exp(B) / (1 + np.exp(B))))
write_script_site1(write[0],write[1],101)
np.savetxt('Site1_param101.txt',write)

In [ ]:
res=pd.read_csv('Site1_param101/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:233]
a=(res.iloc[988,2:]-res.iloc[7714,2:]).values*1000
b=(res.iloc[2732,2:]-res.iloc[7714,2:]).values*1000
c=(res.iloc[6917,2:]-res.iloc[7714,2:]).values*1000
V_model=np.hstack((a,b,c)).reshape(-1,1)

In [ ]:
plt.plot(V_model)
plt.plot(V_obs)

In [ ]:
r = V_model - V_obs.reshape(-1,1);
cost = np.linalg.norm(r)**2
print(f"Cost iter 2= {cost:e}")